# Vehicle Energy & Telemetry Prediction — Final Workflow

This notebook contains the final vehicle-disjoint workflow. Earlier random-row experiments and large model exports were removed. Outputs are intentionally cleared; reported test results in README were supplied from the completed Kaggle run, not recomputed during packaging.

Attach the Kaggle dataset before running all cells. Training reads the full source CSV but models a 200,000-row sample. Target unit and derivation are unverified. This is contemporaneous target estimation, not future forecasting.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import FileLink, display


## Prepare the raw sample


In [ ]:
# 1. Ham veriden temiz bir başlangıç örneklemi oluştur

features = [
    "Vehicle Speed[km/h]",
    "MAF[g/sec]",
    "Engine RPM[RPM]",
    "Absolute Load[%]",
    "OAT[DegC]",
    "Elevation Smoothed[m]",
    "Gradient",
    "Speed Limit[km/h]"
]

target = "Energy_Consumption"

file_path = (
    "/kaggle/input/datasets/yashdev01/"
    "vehicle-energy-and-telemetry-dataset/vehicle_data.csv"
)

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

required_columns = features + [target, "VehId", "Trip"]

parts = []

for chunk in pd.read_csv(
    file_path,
    usecols=required_columns,
    chunksize=250_000
):
    # Hedefi eksik kayıtları çıkar; özelliklerdeki eksiklere henüz dokunma
    parts.append(chunk.dropna(subset=[target]))

raw_sample = (
    pd.concat(parts, ignore_index=True)
    .sample(n=200_000, random_state=42)
    .reset_index(drop=True)
)

del parts

raw_sample["Speed Limit[km/h]"] = pd.to_numeric(
    raw_sample["Speed Limit[km/h]"],
    errors="coerce"
)

print("Örneklem boyutu:", raw_sample.shape)
print("Farklı araç sayısı:", raw_sample["VehId"].nunique())

print("\nÖzelliklerdeki eksik değer sayıları:")
print(raw_sample[features].isna().sum())

print("\nMotor yükü için en yüksek 10 değer:")
print(raw_sample["Absolute Load[%]"].nlargest(10))

## Vehicle-disjoint evaluation


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

X = raw_sample[features]
y = raw_sample[target]
groups = raw_sample["VehId"]

# Aynı araç eğitim ve testte bulunmayacak
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]
y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

train_vehicles = set(groups.iloc[train_idx])
test_vehicles = set(groups.iloc[test_idx])

print("Eğitim kayıt sayısı:", len(train_idx))
print("Test kayıt sayısı:", len(test_idx))
print("Eğitim araç sayısı:", len(train_vehicles))
print("Test araç sayısı:", len(test_vehicles))
print("Ortak araç sayısı:", len(train_vehicles & test_vehicles))

# Medyanlar yalnızca eğitim verisinden öğrenilecek
baseline_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression())
])

final_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=15,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

results = []

for name, pipeline in [
    ("Linear Regression", baseline_pipeline),
    ("Random Forest", final_pipeline)
]:
    pipeline.fit(X_train_group, y_train_group)
    predictions = pipeline.predict(X_test_group)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_group, predictions),
        "RMSE": mean_squared_error(
            y_test_group, predictions
        ) ** 0.5,
        "R2": r2_score(y_test_group, predictions)
    })

print("\nYeni değerlendirme sonuçları:")
print(pd.DataFrame(results).to_string(index=False))

## Save and verify the pipeline


In [ ]:
import joblib
import os
import numpy as np

# Önceki model dosyasını koruyarak yeni pipeline'ı kaydet
model_path = str(output_dir / "vehicle_energy_pipeline.joblib")

joblib.dump(final_pipeline, model_path, compress=3)

# Dosyadan geri yükle
loaded_pipeline = joblib.load(model_path)

# Kaydetmeden önceki ve sonraki tahminleri karşılaştır
check_data = X_test_group.head(20)

before = final_pipeline.predict(check_data)
after = loaded_pipeline.predict(check_data)

assert np.allclose(before, after), "Tahminler eşleşmedi!"

# Eksik girdileri de işleyebiliyor mu?
missing_example = X_test_group.head(1).copy()
missing_example["MAF[g/sec]"] = np.nan

print("Kaydetme ve geri yükleme kontrolü: BAŞARILI")
print("Dosya:", model_path)
print(
    "Sıkıştırılmış dosya boyutu:",
    round(os.path.getsize(model_path) / (1024 ** 2), 3),
    "MB"
)
print(
    "Eksik MAF ile örnek tahmin:",
    loaded_pipeline.predict(missing_example)[0]
)

# Son modelin değişken önemleri
final_importance = pd.Series(
    loaded_pipeline.named_steps["model"].feature_importances_,
    index=features
).sort_values(ascending=False)

print("\nSon modelin değişken önemleri (%):")
print((final_importance * 100).round(2))

## Training environment


In [ ]:
import sklearn
import pandas
import numpy
import joblib

print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)

## Select an illustrative held-out journey


In [ ]:
# Test kayıtlarına model tahminlerini ekle
test_review = raw_sample.iloc[test_idx].copy()

test_review["Predicted_Energy"] = final_pipeline.predict(
    test_review[features]
)

# Test örnekleminde en fazla kaydı olan araç–yolculuk çifti
trip_counts = (
    test_review.groupby(["VehId", "Trip"])
    .size()
    .sort_values(ascending=False)
)

vehicle_id, trip_id = trip_counts.index[0]

print("Selected held-out vehicle:", vehicle_id, "trip:", trip_id)


## Export the available-target journey records


In [ ]:
# Seçilen araç ve yolculuğun hedefi mevcut tüm kayıtlarını al
trip_parts = []

for chunk in pd.read_csv(
    file_path,
    usecols=features + [
        target, "VehId", "Trip", "Timestamp(ms)"
    ],
    chunksize=250_000
):
    selected = chunk[
        (chunk["VehId"] == vehicle_id) &
        (chunk["Trip"] == trip_id)
    ].dropna(subset=[target])

    if not selected.empty:
        trip_parts.append(selected)

trip_review = pd.concat(trip_parts, ignore_index=True)

trip_review["Speed Limit[km/h]"] = pd.to_numeric(
    trip_review["Speed Limit[km/h]"],
    errors="coerce"
)

trip_review = trip_review.sort_values(
    "Timestamp(ms)"
).reset_index(drop=True)

trip_review["Predicted_Energy"] = final_pipeline.predict(
    trip_review[features]
)

trip_review["Residual"] = (
    trip_review[target] - trip_review["Predicted_Energy"]
)

trip_review["Absolute_Error"] = trip_review["Residual"].abs()

trip_review["Elapsed_Seconds"] = (
    trip_review["Timestamp(ms)"] -
    trip_review["Timestamp(ms)"].min()
) / 1000

trip_review.to_csv(
    str(output_dir / "trip_review.csv"),
    index=False
)

print("Hedefi mevcut yolculuk kayıt sayısı:", len(trip_review))
print(
    "Zaman aralığı (saniye):",
    trip_review["Elapsed_Seconds"].max()
)

display(FileLink(str(output_dir / "trip_review.csv")))
display(FileLink(str(output_dir / "vehicle_energy_pipeline.joblib")))


## Interpretation and limitations

Final reported test results: Linear Regression R² 0.669055; Random Forest MAE 0.000670, RMSE 0.001604, R² 0.897888. The test set contains 62 vehicles absent from training (246 vehicles). Only one split was evaluated.

Target construction and physical unit remain unverified; high MAF importance could partly reflect target derivation. The extreme absolute-load value was retained. No external validation, fault diagnosis, measured savings or live vehicle integration is claimed. The example journey is selected by available test-sample record count; its MAE is not the overall test metric. The CSV includes only records with available target values and may contain time gaps.
